In [ ]:
# mike babb
# created: 2026 08 23
# find five groups of five letters

In [ ]:
# standard
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx

In [ ]:
# define a function to load a pickle
def load_pickle(file_name):
    if os.path.exists(file_name):
        with open(file_name, 'rb') as handle:
            de_pickle = pickle.load(handle)
    else:
        de_pickle = None
        print("file does not exist")
    return de_pickle

In [ ]:
char_matrix = load_pickle(file_name = 'char_matrix.pkl')
letter_dict = load_pickle(file_name = 'letter_dict.pkl')
letter_rank_dict = load_pickle(file_name ='letter_rank_dict.pkl')
word_df = load_pickle(file_name = 'word_df.pkl')

In [ ]:
word_df.head()

In [ ]:
word_df.shape

In [ ]:
# using the word_group, select entries
word_df = word_df.drop_duplicates(subset = ['word_group'])

In [ ]:
word_df = word_df.loc[word_df['n_unique_chars'] == 5, :].reset_index(drop = True)

In [ ]:
word_df.head()

In [ ]:
word_df['word_id'] = range(0, word_df.shape[0])

In [ ]:
word_df['lcase'].map(lambda x: len(set(x))).describe()

In [ ]:
# build a char_matrix
char_matrix = np.zeros(shape = (word_df.shape[0], 26), dtype = np.int8)
def build_char_matrix(row):
    for l in row['lcase']:
        char_matrix[row['word_id'], letter_dict[l]] += 1

    


In [ ]:
outcome = word_df.apply(build_char_matrix, axis = 1)

In [ ]:
char_matrix.max()

In [ ]:
assert word_df.shape[0] == char_matrix.shape[0]

In [ ]:
def add_that_shit(m1, m2):
    ps = m1[:, None, :] + m2[None, :, :]
    matching_pairs = np.argwhere(ps.max(axis = 2) == 1)
    matching_pairs = matching_pairs[:int(matching_pairs.shape[0] /  2) , :]       
    outcome = m1[matching_pairs[:, 0]] + m2[matching_pairs[:, 1]]
    return outcome, matching_pairs
    


In [ ]:
r1, mp1 = add_that_shit(m1 = char_matrix, m2 = char_matrix)

In [ ]:
r1.shape

In [ ]:
r_list = []
mp_list = []
chunker1 = np.arange(start = 0, stop = r1.shape[0] + 50000,step = 50000 )
for icc, curr_chunk in enumerate(chunker1[:-1]):    
    
    next_chunk = chunker1[icc + 1]
    print('l1', curr_chunk, next_chunk)
    r2, mp2 = add_that_shit(m1 = r1[curr_chunk:next_chunk, :], m2 = char_matrix)
    print('l1', r2.shape)
    r_list.append(r2)
    mp_list.append(mp2)


In [ ]:

    chunker2 = np.arange(start = 0, stop = r2.shape[0] + 10000,step = 10000 )
    for icc2, cc2 in enumerate(chunker2[:-1]):
        nc2 = chunker2[icc2 + 1]
        print('l2', cc2, nc2)
        r3, mp3 = add_that_shit(m1 = r2[cc2:nc2, :], m2 = char_matrix)
        print('l2', r3.shape)

        chunker3 = np.arange(start = 0, stop = r3.shape[0] + 10000,step = 10000)
        for icc3, cc3 in enumerate(chunker3[:-1]):
            nc3 = chunker3[icc3 + 1]
            print('l3', cc3, nc3)
            r4, mp4 = add_that_shit(m1 = r3[cc3:nc3, :], m2 = char_matrix)
            print('l4', r4.shape)
            if r4.size != 0:
                r_list.append([r1, r2, r3, r4])
                mp_list.append([mp1, mp2, mp3, mp4])
    


In [ ]:
word_df['lcase_set'] = word_df['lcase'].map(lambda x: set(x))
word_df['lcase_tuple'] = word_df['lcase_set'].map(lambda x: tuple(x))

In [ ]:
# build dictionaries
word_dict = {}
for i_row, my_row in word_df.iterrows():
    word_dict[my_row['lcase']] = my_row['lcase_set']

In [ ]:
r1 = add_that_shit(m1 = char_matrix, m2 = char_matrix)

In [ ]:
edge_list = []
for lc0, lc1 in combinations(word_df['lcase'], 2):
    if word_dict[lc0].isdisjoint(word_dict[lc1]):
        #edge_list.append([ms[0], ms[1]])
        myg.add_edge(u_of_edge=lc0, v_of_edge=lc1)


In [ ]:
import networkx as nx
from itertools import combinations

In [ ]:
myg = nx.Graph()
edge_list = []
for lc0, lc1 in combinations(word_df['lcase'], 2):
    if word_dict[lc0].isdisjoint(word_dict[lc1]):
        #edge_list.append([ms[0], ms[1]])
        myg.add_edge(u_of_edge=lc0, v_of_edge=lc1)


In [ ]:
for r0 in char_matrix:
    

In [ ]:
import scipy

In [ ]:
scipy.special.comb(word_df.shape[0], k = 5, exact=True)

In [ ]:
# chunks of 10K
chunker = range(0, r1.shape[0] + 10000, 10000)
list(chunker)


In [ ]:
# get the first words

In [ ]:
lm0 = char_matrix.copy()

In [ ]:
grand_output_list = []
for ii, l0 in enumerate(lm0[:10, :]):
    print(ii, l0)    
    outcome = ((lm0 + l0) <= 1).all(axis = 1)
    lm1 = lm0[outcome, :]
    for l1 in lm1:
        outcome = ((lm1 + l1) <= 1).all(axis = 1)
        lm2 = lm1[outcome, :]
        for l2 in lm2:
            outcome = ((lm2 + l2) <= 1).all(axis = 1)
            lm3 = lm2[outcome, :]
            for l3 in lm3:
                outcome = ((lm3 + l3) <= 1).all(axis = 1)
                lm4 = lm3[outcome, :]
                if lm4.shape[0] > 0:
                    grand_output_list.append([l0, l1, l2, l3, l4])

In [ ]:
ncm = char_matrix[outcome, :]
for irow in ncm:
    



In [ ]:
testo

In [ ]:
# thinking about this wrong... it's a case of letters left....

In [ ]:
word_df.head()

In [ ]:
word_df['lcase_set'] = word_df['lcase'].map(lambda x: set(x))
word_df['lcase_tuple'] = word_df['lcase_set'].map(lambda x: tuple(x))

In [ ]:
# build dictionaries
word_dict = {}
for i_row, my_row in word_df.iterrows():
    word_dict[my_row['lcase']] = my_row['lcase_set']

In [ ]:
import networkx as nx
from itertools import combinations

In [ ]:
myg = nx.Graph()
edge_list = []
for lc0, lc1 in combinations(word_df['lcase'], 2):
    if word_dict[lc0].isdisjoint(word_dict[lc1]):
        #edge_list.append([ms[0], ms[1]])
        myg.add_edge(u_of_edge=lc0, v_of_edge=lc1)


In [ ]:
myg.number_of_edges()

In [ ]:
word_df.head()

In [ ]:
nx.__version__

In [ ]:
#myg_backup = myg.copy()

In [ ]:
# get 100 words
test_list = word_df['lcase'].tolist()[:10]

In [ ]:
test_list

In [ ]:
#myg = myg_backup.copy()
output_list = []
for w0 in word_df['lcase']:    
#for w0 in test_list:
    reject_groupings = set()
    print(w0)
    l0 = [w0, '', '', '', '']
    #output_list.append(l0)        
    # this is immediately adjacent
    w1_set = set(nx.neighbors(G = myg, n = w0))
    fl0 = set(w0)    
    for w1 in w1_set:
        if reject_groupings.isdisjoint(permutations((w0, w1), r = 2)):
            if fl0.isdisjoint(set(w1)):
                fl1 = fl0.copy()
                fl1.update(w1)                 
                l1 = l0[:]
                l1[1] = w1
                #output_list.append(l1)                
                # immediately adjacent candidates        
                w2_set = set(nx.neighbors(G = myg, n = w1))
                w2_set = w2_set.intersection(w1_set)                
                if w2_set:
                    for w2 in w2_set:
                        if reject_groupings.isdisjoint(permutations((w0, w1,w2), r = 2)):
                            if fl1.isdisjoint(set(w2)):
                                fl2 = fl1.copy()
                                fl2.update(w2)
                                l2 = l1[:]
                                l2[2] = w2
                                #output_list.append(l2)                        
                                w3_set = set(nx.neighbors(G = myg, n = w2))                        
                                w3_set = w3_set.intersection(w2_set)                                
                                if w3_set:
                                    for w3 in w3_set:
                                        if reject_groupings.isdisjoint(permutations((w0, w1, w2, w3), r = 2)):                                        
                                            if fl2.isdisjoint(set(w3)):
                                                fl3 = fl2.copy()
                                                fl3.update(w3)
                                                l3 = l2[:]
                                                l3[3] = w3
                                                #output_list.append(l3)    
                                                # print(l3)
                                                w4_set = set(nx.neighbors(G = myg, n = w3))
                                                w4_set = w4_set.intersection(w3_set)
                                                if w4_set:
                                                    for w4 in w4_set:
                                                        if reject_groupings.isdisjoint(permutations((w0, w1, w2, w3, w4), r = 2)):
                                                            if fl3.isdisjoint(set(w4)):
                                                                fl4 = fl3.copy()
                                                                fl4.update(w4)
                                                                l4 = l3[:]
                                                                l4[4] = w4
                                                                output_list.append(l4)    
                                                                print(l4)
                                                else:                                                    
                                                    # compute combos                                                                                                    
                                                    #print(l3)
                                                    #print(w0, w1, w2, w3)
                                                    perms = list(permutations(l3, r = 2))
                                                    #print(perms)
                                                    reject_groupings.update(perms)
                                                    myg.remove_edges_from(perms)


#print('yay')  
#reject_groupings       

In [ ]:
output_list

In [ ]:
len(reject_groupings)

In [ ]:
myg.number_of_edges()

In [ ]:
output_list

In [ ]:
w3_seta

In [ ]:
reject_groupings

In [ ]:
len(testo)

In [ ]:
next_nodes

In [ ]:
myg.nodes(data = 'abhor')

In [ ]:
# this needs to be a recursive function


In [ ]:
test_outcome = np.where(tv1 == 0)[0]

In [ ]:
test_outcome

In [ ]:
outcome = char_matrix - tv1

In [ ]:
outcome

In [ ]:
outcome1 = outcome

In [ ]:
for wid in word_df['word_id'].tolist():
    cw = char_matrix[wid, :]
    
    # t/f list
    outcome = np.abs(char_matrix - cw).sum(1) == 10
    # next word id list
    wil2 = wil1[outcome]
    if wil2.shape[0] > 
    # next char matrix
    cm2 = cm1[outcome, :]
    
    
    # I'll need five different stages of the word_id_list 

    # 


    
    

In [ ]:
)

In [ ]:
tv1 = char_matrix[0, :]

In [ ]:
tv2 = char_matrix[-1, :]

In [ ]:
tv3 = tv2 - tv1

In [ ]:
testo = char_matrix - tv1

In [ ]:
testo

In [ ]:
testo2 = testo[np.abs(testo).sum(1) == 10, :]

In [ ]:
testo2

In [ ]:
# step 2
testo = testo[testo.sum(0)  ]

In [ ]:
# for each row in the char_matrix:
# substract from the chart_matrix:

In [ ]:
np.abs(tv3).sum()

In [ ]:
word_df.iloc[0]

In [ ]:
word_df.iloc[-1]